In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Diretórios base do projeto
project_root = Path("..").resolve()
processed_path = project_root / "data" / "processed"
output_path = project_root / "data" / "output"
output_path.mkdir(parents=True, exist_ok=True)

# Carregando dataframes processados
# Cada arquivo já foi tratado no pipeline anterior; aqui apenas consolidamos a análise.
df_orders = pd.read_csv(processed_path / "orders_processed.csv", encoding="UTF-8")
df_items = pd.read_csv(processed_path / "order_items_processed.csv", encoding="UTF-8")
df_payments = pd.read_csv(processed_path / "order_payments_processed.csv", encoding="UTF-8")
df_reviews = pd.read_csv(processed_path / "order_reviews_processed.csv", encoding="UTF-8")
df_products = pd.read_csv(processed_path / "products_processed.csv", encoding="UTF-8")
df_sellers = pd.read_csv(processed_path / "sellers_processed.csv", encoding="UTF-8")
df_customers = pd.read_csv(processed_path / "customers_processed.csv", encoding="UTF-8")

In [5]:
# Bloco 1: conversão de colunas temporais para facilitar dashboards e agregações mensais
# A padronização do tipo datetime é essencial para análise temporal e cálculo de SLA.
for df, datetime_cols in [
    (df_orders, ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]),
    (df_items, ["shipping_limit_date"]),
    (df_reviews, ["review_creation_date", "review_answer_timestamp"]),
]:
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Extração de componentes de data para uso em Tableau e filtros
# Isso permite responder perguntas como: receita por mês, pedidos por dia da semana e atraso por região.
df_orders = df_orders.assign(
    purchase_year=df_orders["order_purchase_timestamp"].dt.year,
    purchase_month=df_orders["order_purchase_timestamp"].dt.month,
    purchase_month_name=df_orders["order_purchase_timestamp"].dt.month_name(locale="pt_BR"),
    purchase_day=df_orders["order_purchase_timestamp"].dt.day,
    purchase_dayofweek=df_orders["order_purchase_timestamp"].dt.dayofweek,
    purchase_weekday_name=df_orders["order_purchase_timestamp"].dt.day_name(),
    purchase_hour=df_orders["order_purchase_timestamp"].dt.hour,
    approval_year=df_orders["order_approved_at"].dt.year,
    approval_month=df_orders["order_approved_at"].dt.month,
    approval_day=df_orders["order_approved_at"].dt.day,
    carrier_year=df_orders["order_delivered_carrier_date"].dt.year,
    carrier_month=df_orders["order_delivered_carrier_date"].dt.month,
    carrier_day=df_orders["order_delivered_carrier_date"].dt.day,
    delivered_year=df_orders["order_delivered_customer_date"].dt.year,
    delivered_month=df_orders["order_delivered_customer_date"].dt.month,
    delivered_day=df_orders["order_delivered_customer_date"].dt.day,
    estimated_year=df_orders["order_estimated_delivery_date"].dt.year,
    estimated_month=df_orders["order_estimated_delivery_date"].dt.month,
    estimated_day=df_orders["order_estimated_delivery_date"].dt.day,
)

# Corrigindo possível incompatibilidade de locale em alguns ambientes.
# Se necessário, a coluna de nome do mês pode ser usada sem depender do locale.
df_orders["purchase_month_name"] = df_orders["order_purchase_timestamp"].dt.strftime("%b")

# Cálculo de métricas de logística e satisfação
# Estas colunas ajudam na análise de atraso e relação com avaliações.
df_orders["delivery_days"] = (
    (df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df_orders["estimated_delivery_days"] = (
    (df_orders["order_estimated_delivery_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df_orders["delay_days"] = (
    (df_orders["order_delivered_customer_date"] - df_orders["order_estimated_delivery_date"]).dt.total_seconds() / 86400
)

df_orders["is_delayed"] = (df_orders["delay_days"] > 0).astype(int)

df_orders["delay_flag"] = np.where(df_orders["delay_days"] > 0, "Atrasado", "NoPrazo")

In [6]:
# Bloco 2: agregação de pagamentos por pedido para facilitar a análise financeira
# A chave de junção é order_id; cada pedido pode ter um ou mais pagamentos.
df_payments_agg = (
    df_payments.groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum"),
        payment_methods_count=("payment_type", "nunique"),
        payment_types=("payment_type", lambda s: ", ".join(s.dropna().astype(str).unique()))
    )
)

In [7]:
# Bloco 3: agregação de itens por pedido para compor a visão de Receita e volume por pedido
# Isso permite calcular ticket médio, itens por pedido e valor bruto/logístico.
df_items_agg = (
    df_items.groupby("order_id", as_index=False)
    .agg(
        items_quantity=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
)

In [8]:
# Bloco 4: agregação de avaliações por pedido para responder relação entre qualidade e logística
# Como cada pedido pode ter eventualmente uma avaliação, usamos left join preservando pedidos sem revisão.
df_reviews_agg = (
    df_reviews.groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_max=("review_score", "max"),
        review_score_min=("review_score", "min"),
        review_count=("review_id", "count")
    )
)

In [9]:
# Bloco 5: joins principais do modelo analítico
# A abordagem de camada analítica mantém os joins organizados por responsabilidade.
# 5.1 Pedido + cliente
orders_customers = df_orders.merge(df_customers, on="customer_id", how="left", suffixes=("_order", "_customer"))

# 5.2 Pedido + itens agregados
orders_items = orders_customers.merge(df_items_agg, on="order_id", how="left")

# 5.3 Pedido + pagamentos agregados
orders_items_payments = orders_items.merge(df_payments_agg, on="order_id", how="left")

# 5.4 Pedido + avaliações agregados
orders_items_payments_reviews = orders_items_payments.merge(df_reviews_agg, on="order_id", how="left")

# 5.5 Pedido + produto + vendedor
# Aqui usamos order_id como base para enriquecer com produto e vendedor de cada item.
# Como um pedido pode ter múltiplos itens, o join direto em df_items preserva a granularidade de item.
model_df = orders_items_payments_reviews.merge(df_items, on="order_id", how="left")
model_df = model_df.merge(df_products, on="product_id", how="left", suffixes=("_item", "_product"))
model_df = model_df.merge(df_sellers, on="seller_id", how="left", suffixes=("_item", "_seller"))

# Ajustes de tipos numéricos e preenchimento
model_df["payment_total"] = model_df["payment_total"].fillna(0)
model_df["total_price"] = model_df["total_price"].fillna(0)
model_df["total_freight"] = model_df["total_freight"].fillna(0)
model_df["review_score_mean"] = model_df["review_score_mean"].fillna(0)
model_df["review_count"] = model_df["review_count"].fillna(0)

# Como o schema processado não possui a coluna `quantity`, representamos a unidade do item por linha.
# Isso mantém a granularidade de item e permite calcular a receita por linha do pedido.
model_df["quantity"] = 1
model_df["item_revenue"] = model_df["price"] * model_df["quantity"]

In [10]:
# Bloco 6: exportação dos datasets prontos para o Tableau
# Os CSVs aqui são gerados para atender às perguntas de negócio do arquivo de entendimento.
model_df.to_csv(output_path / "orders_customer_product_seller_full.csv", index=False)
orders_items_payments_reviews.to_csv(output_path / "orders_customer_financial_analytics.csv", index=False)
df_orders.to_csv(output_path / "orders_temporal_features.csv", index=False)

display(model_df.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,quantity,item_revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,Utilidades_Domesticas,500.0,19.0,8.0,13.0,9350.0,Maua,SP,1,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,Perfumaria,400.0,19.0,13.0,19.0,31570.0,Belo Horizonte,SP,1,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,Automotivo,420.0,24.0,19.0,21.0,14840.0,Guariba,SP,1,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,Pet_Shop,450.0,30.0,10.0,20.0,31842.0,Belo Horizonte,MG,1,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,Papelaria,250.0,51.0,15.0,15.0,8752.0,Mogi Das Cruzes,SP,1,19.90


In [14]:
# Bloco final: base analítica pronta para o Tableau
# Objetivo: entregar uma tabela de fácil consumo para dashboards, com dimensões de negócio e KPIs já preparados.

# 1. Base de pedido: 1 linha por pedido, ideal para visão macro e KPIs
# Reaproveita a tabela de pedidos já enriquecida com cliente, pagamentos e avaliações.
tableau_order_base = orders_items_payments_reviews.copy()

tableau_order_base["customer_region"] = tableau_order_base["customer_state"].astype(str)
tableau_order_base["delivery_status"] = np.where(tableau_order_base["delay_days"] > 0, "Atrasado", "NoPrazo")
tableau_order_base["avg_review_score"] = tableau_order_base["review_score_mean"].fillna(0)
tableau_order_base["revenue_total"] = tableau_order_base["payment_total"].fillna(0)
tableau_order_base["ticket_medio"] = np.where(
    tableau_order_base["items_quantity"].fillna(0) > 0,
    tableau_order_base["payment_total"] / tableau_order_base["items_quantity"],
    0
)
tableau_order_base["order_month_name"] = tableau_order_base["purchase_month_name"]
tableau_order_base["order_weekday"] = tableau_order_base["purchase_weekday_name"]
tableau_order_base["order_year"] = tableau_order_base["purchase_year"]

tableau_order_base = tableau_order_base.sort_values("order_purchase_timestamp")

# 2. Base detalhada por item: 1 linha por item, útil para análises de categoria, seller e produto
# Mantém a granularidade mais rica para filtros e drilldown.
tableau_item_base = model_df.copy()
tableau_item_base["customer_region"] = tableau_item_base["customer_state"].astype(str)
tableau_item_base["seller_region"] = tableau_item_base["seller_state"].astype(str)
tableau_item_base["delivery_status"] = np.where(tableau_item_base["delay_days"] > 0, "Atrasado", "NoPrazo")
tableau_item_base["item_revenue"] = tableau_item_base["price"] * tableau_item_base["quantity"]

# 3. Exportação final em CSV
# Esses arquivos serão consumidos diretamente pelo Tableau.
tableau_order_base.to_csv(output_path / "tableau_order_base.csv", index=False)
tableau_item_base.to_csv(output_path / "tableau_item_base.csv", index=False)

display(tableau_order_base.head())
display(tableau_item_base.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,review_score_min,review_count,customer_region,delivery_status,avg_review_score,revenue_total,ticket_medio,order_month_name,order_weekday,order_year
4541,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,Shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaT,2016-10-20,2016,9,...,1.0,1.0,RR,NoPrazo,1.0,136.23,68.115,Sep,Sunday,2016
4396,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,Canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,NaT,NaT,2016-10-28,2016,9,...,1.0,1.0,RS,NoPrazo,1.0,75.06,75.060,Sep,Monday,2016
10071,809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,Canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,NaT,NaT,2016-09-30,2016,9,...,1.0,1.0,SP,NoPrazo,1.0,40.95,0.000,Sep,Tuesday,2016
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,Delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,2016,9,...,1.0,1.0,SP,Atrasado,1.0,0.00,NaN,Sep,Thursday,2016
83078,71303d7e93b399f5bcd537d124c0bcfa,b106b360fe2ef8849fbbd056f777b4d5,Canceled,2016-10-02 22:07:52,2016-10-06 15:50:56,NaT,NaT,2016-10-25,2016,10,...,1.0,1.0,SP,NoPrazo,1.0,109.34,109.340,Oct,Sunday,2016


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,quantity,item_revenue,customer_region,seller_region,delivery_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,8.0,13.0,9350.0,Maua,SP,1,29.99,SP,SP,NoPrazo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,13.0,19.0,31570.0,Belo Horizonte,SP,1,118.70,BA,SP,NoPrazo
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,19.0,21.0,14840.0,Guariba,SP,1,159.90,GO,SP,NoPrazo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,10.0,20.0,31842.0,Belo Horizonte,MG,1,45.00,RN,MG,NoPrazo
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,15.0,15.0,8752.0,Mogi Das Cruzes,SP,1,19.90,SP,SP,NoPrazo


In [13]:
# Diagnóstico de colunas da base analítica
print('Colunas em orders_items_payments_reviews:')
print(orders_items_payments_reviews.columns.tolist()[:80])
print('\nColunas em model_df:')
print(model_df.columns.tolist()[:80])

Colunas em orders_items_payments_reviews:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_year', 'purchase_month', 'purchase_month_name', 'purchase_day', 'purchase_dayofweek', 'purchase_weekday_name', 'purchase_hour', 'approval_year', 'approval_month', 'approval_day', 'carrier_year', 'carrier_month', 'carrier_day', 'delivered_year', 'delivered_month', 'delivered_day', 'estimated_year', 'estimated_month', 'estimated_day', 'delivery_days', 'estimated_delivery_days', 'delay_days', 'is_delayed', 'delay_flag', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'items_quantity', 'total_price', 'total_freight', 'payment_total', 'payment_methods_count', 'payment_types', 'review_score_mean', 'review_score_max', 'review_score_min', 'review_count']

Colunas em model_df:
['order_id', 'customer_id', 'order_status',